# 14. Ders: Yapay Zeka ile Akıllı Metin Özetleme (Kariyer Terzisi - Bölüm 2)

## Adım 1: Teorik Çerçeve ve Günlük Hayat Analojisi

Bir önceki derste, metinlerin içindeki anahtar kelimeleri kelime kelime çekip çıkarmayı öğrendik. Ancak bazen bir makalenin, 10 sayfalık bir toplantı raporunun veya upuzun bir özgeçmişin 'ne anlattığını' bilmemiz gerekir. Bir insana uzun bir rapor verip "Bunu bana özetle" dediğinizde, insan rapordaki cümleleri birebir kes-yapıştır yapmaz; konuyu anlar ve kendi kelimeleriyle çok daha kısa, yeni bir paragraf yazar.

Yapay zekanın da uzun metinleri kendi kendine yeniden yorumlayarak kısaltmasına 'Metin Özetleme' (Text Summarization) denir. Bu dersimizde Hugging Face'in özetleme modellerini kullanarak devasa metinleri birkaç cümleye nasıl sıkıştıracağımızı öğreneceğiz.

## Adım 2: Teknik Terimler ve Sözlük

- Summarization (Özetleme): Yapay zekanın uzun bir metnin ana fikrini koruyarak daha kısa bir metin oluşturması görevi.
- Extractive Summarization (Çıkarımsal Özetleme): Modelin, metindeki en önemli cümleleri olduğu gibi kopyalayıp alt alta yapıştırarak oluşturduğu ilkel özetleme yöntemidir.
- Abstractive Summarization (Yorumlayıcı Özetleme): Modelin tıpkı bir insan gibi metni okuyup, tamamen yeni cümleler kurarak oluşturduğu gelişmiş (GPT/T5 türevi) özetleme yöntemidir.
- max_length (Maksimum Uzunluk): Üretilecek özetin çıkabileceği en fazla kelime (token) sınırı.
- min_length (Minimum Uzunluk): Üretilecek özetin kesinlikle ulaşması gereken en az kelime (token) sınırı.

## Adım 3: Hugging Face Summarization Pipeline Çağırma

Önceki projelerde (MoodJar) `sentiment-analysis` aracını kullanmıştık. Şimdi görev adımızı `summarization` olarak değiştireceğiz.

In [ ]:
from transformers import pipeline

# Özetleme (Summarization) pipeline'ını çağırıyoruz.
# Varsayılan olarak DistilBART isimli devasa bir İngilizce özetleme modeli yükleyecektir.
# Not: Dosya boyutu büyük olabilir, ilk çalışmada Colab'in indirmesini bekleyin.
ozetleyici_model = pipeline("summarization")

print("Yapay Zeka Özetleme Motoru başarıyla yüklendi.")

## Adım 4: Özetlenecek Uzun Metni Tanımlama

Yapay Zeka hakkında İngilizce, oldukça uzun ve ansiklopedik bir bilgi paragrafı tanımlayalım.

In [ ]:
uzun_makale = """
Artificial intelligence (AI) is intelligence demonstrated by machines, as opposed to the natural intelligence displayed by animals including humans. 
Leading AI textbooks define the field as the study of "intelligent agents": any system that perceives its environment and takes actions that maximize its chance of achieving its goals. 
Some popular accounts use the term "artificial intelligence" to describe machines that mimic "cognitive" functions that humans associate with the human mind, such as "learning" and "problem-solving", however, this definition is rejected by major AI researchers. 
AI applications include advanced web search engines, recommendation systems (used by YouTube, Amazon, and Netflix), understanding human speech (such as Siri and Alexa), self-driving cars (e.g., Tesla), automated decision-making and competing at the highest level in strategic game systems.
"""

# Metnin orijinal kelime sayısını bulalım
orijinal_kelime_sayisi = len(uzun_makale.split())
print(f"Orijinal metnimizin kelime sayısı: {orijinal_kelime_sayisi}")

## Adım 5: Varsayılan Ayarlarla İlk Özeti Alma

Makaleyi modelimize gönderip ne anlayıp çıkardığına bakalım.

In [ ]:
# Model uzun makaleyi okur ve bir liste içinde sözlük döndürür.
ozet_sonucu = ozetleyici_model(uzun_makale)

# 'summary_text' anahtarını çekerek özeti alıyoruz
uretilen_ozet = ozet_sonucu[0]['summary_text']

print("--- YAPAY ZEKA ÖZETİ ---")
print(uretilen_ozet)

## Adım 6: Özetin Uzunluğunu Kontrol Etmek

Bazen Twitter (X) gibi platformlar için çok kısa bir özet isteriz (Örn: En fazla 30 token). Bazen de kitabın arkasına yazılacak daha uzun bir arka kapak yazısı isteriz. Modelin ne kadar uzun konuşacağına `max_length` ve `min_length` parametreleriyle karar veririz.

In [ ]:
# Kural: min_length, max_length'ten kesinlikle küçük olmalıdır.
# Kural 2: İstenen max_length, orijinal metnin token sayısından çok büyük olmamalıdır.

kisa_ozet_sonucu = ozetleyici_model(uzun_makale, max_length=45, min_length=15, do_sample=False)

print("--- KISA ÖZET (Maksimum 45 Token) ---")
print(kisa_ozet_sonucu[0]['summary_text'])

## Adım 7: Orijinal Metin ile Özet Arasındaki Karakter Tasarrufunu Ölçme

Yaptığımız sistemin kullanıcıya (örneğin Kariyer Terzisi projesinde İK uzmanına) ne kadar zaman kazandırdığını matematiksel olarak hesaplayalım.

In [ ]:
kisa_ozet_metni = kisa_ozet_sonucu[0]['summary_text']

orijinal_uzunluk = len(uzun_makale)
ozet_uzunlugu = len(kisa_ozet_metni)

tasarruf_orani = 100 - ((ozet_uzunlugu / orijinal_uzunluk) * 100)

print(f"Orijinal Karakter Sayısı: {orijinal_uzunluk}")
print(f"Özet Karakter Sayısı: {ozet_uzunlugu}")
print(f"\nYapay zeka sayesinde okunacak metin yükünden Yüzde {tasarruf_orani:.1f} tasarruf edildi!")

## Adım 8: Türkçe Özetleme (Multilingual Modeller) Üzerine

İngilizce (BART veya T5) modelleri Türkçe verdiğiniz metinleri çok kötü özetler veya hata verir. Türkçe için genellikle `mT5` (Multilingual T5) veya geliştiricilerin eğittiği özel Türkçe modeller (Örn: `mrm8488/bert2bert_shared-turkish-summarization`) kullanılır. Colab'de bu modellerin boyutu çok büyük (Bazen 2-3 GB) olabildiği için, gerçek hayatta profesyonel veri bilimciler bu modelleri Colab'in GPU (Donanım hızlandırıcısı) ile çalıştırır.

*Not: Sınıf ortamındaki eğitimlerde genellikle İngilizce modeller üzerinden mantık kavratılır.*

## Adım 9: Dene ve Keşfet - Kendi İngilizce Metnini Özetlet

Adım 4'te yer alan `uzun_makale` değişkenindeki metni silin. İnternetten İngilizce bir gazete haberi (BBC, CNN vb.) bulun ve o metni yapıştırıp hücreleri baştan sona çalıştırın. Modelin ana konuyu ne kadar iyi tespit ettiğini gözlemleyin.

## Adım 10: Sıra Sende Görevi - Haber Sitesi Otomatik Özet Botu

Problem: Bir haber sitesi editörüsünüz. Sitenize giren uzun haber makalelerinin en üstüne '3 Cümlede Özet' başlıklı bir bölüm eklemek istiyorsunuz.

Gereksinimler:
1. `haber_ozetleyici(makale_metni)` adında bir fonksiyon oluşturun.
2. Fonksiyon dışarıdan aldığı metni `ozetleyici_model` içine koysun.
3. Siteniz çok uzun özetler istemediği için `max_length=50` sınırını koysun.
4. Çıkan özet metnini `return` komutu ile geri döndürsün.
5. Dışarıda fonksiyonu test etmek için kendi yazdığınız veya internetten bulduğunuz uzun bir haberi fonksiyona gönderin ve sonucu `print()` ile yazdırın.

In [ ]:
# GÖREV KODLARINIZI BURAYA YAZINIZ:



Tebrikler! 14. Dersi tamamlayarak yapay zekanın sadece metni analiz etmediğini, metnin bağlamını anlayarak onu insan gibi yeniden (abstractive) sentezleyebildiğini kanıtladınız.

Bir sonraki dersimizde bu sentezleme yeteneğini bir adım öteye taşıyacağız ve modelden bir özet değil, daha önce dünyada hiç var olmamış bir metin (Örneğin bir şiir veya hikaye) üretmesini isteyeceğimiz 'Yaratıcı Kelimeler' projesine geçeceğiz.